# 🗄️ Python & Databases: MySQL · SQLite · PostgreSQL
## A School Data Analysis Tutorial

---

This notebook teaches you how to connect Python to three mainstream databases,
using **real school data** (students, teachers, grades) throughout.

| Section | Database | Use case |
|---------|----------|----------|
| 1 | **MySQL** | Main school MIS — create, populate, query |
| 2 | **SQLite** | Quick local analysis — no server needed |
| 3 | **PostgreSQL** | Enterprise / cloud analytics |

> 💡 The SQL you write is nearly identical across all three — only the *connection* syntax changes.


In [2]:
from IPython.display import HTML, display
display(HTML('''
<div style='display:flex;align-items:center;gap:16px;padding:14px 20px;
            background:#f0f4ff;border-left:6px solid #3B6AC4;border-radius:8px;margin:8px 0'>
  <svg xmlns='http://www.w3.org/2000/svg' width='48' height='48' viewBox='0 0 48 48'>
    <rect width='48' height='48' rx='8' fill='#3B6AC4'/>
    <text x='24' y='34' font-size='26' text-anchor='middle' fill='white' font-family='Arial'>🗄</text>
  </svg>
  <div>
    <div style='font-size:22px;font-weight:bold;color:#1a1a2e'>Part 1 — MySQL</div>
    <div style='font-size:13px;color:#555;margin-top:2px'>
      Install · Connect · Create database · Design schema · Load CSV data · Query · Update · Delete
    </div>
  </div>
</div>
'''))


## 1.1 · Installation

Install the MySQL connector once, then import it in every subsequent cell.

**Download MySQL Server (free):**
- Windows: https://dev.mysql.com/downloads/installer/
- macOS: https://dev.mysql.com/downloads/mysql/
- MySQL Workbench (GUI): https://dev.mysql.com/downloads/workbench/


In [3]:
# Run once to install the Python ↔ MySQL bridge library
# !pip install mysql-connector-python

from getpass import getpass           # hides password as you type
from mysql.connector import connect, Error
import pandas as pd

print("✅ Libraries ready")


✅ Libraries ready


## 1.2 · Connect to MySQL

We use a `with` block — Python closes the connection automatically when the block ends,
even if an error occurs.


In [4]:
from IPython.display import HTML, display
display(HTML('''
<div style='background:#fffbe6;border:1px solid #f0c040;border-radius:8px;padding:12px 18px;margin:6px 0'>
  <b>🔑 How a MySQL connection works</b><br>
  <svg xmlns='http://www.w3.org/2000/svg' width='480' height='70' style='margin-top:10px'>
    <rect x='0'   y='20' width='110' height='32' rx='6' fill='#3B6AC4'/>
    <text x='55'  y='41' text-anchor='middle' fill='white' font-size='12' font-family='Arial'>Python script</text>
    <line x1='110' y1='36' x2='175' y2='36' stroke='#333' stroke-width='2' marker-end='url(#a)'/>
    <text x='142' y='28' font-size='10' fill='#555' font-family='Arial'>TCP</text>
    <rect x='175' y='20' width='110' height='32' rx='6' fill='#e67e22'/>
    <text x='230' y='41' text-anchor='middle' fill='white' font-size='12' font-family='Arial'>MySQL Server</text>
    <line x1='285' y1='36' x2='350' y2='36' stroke='#333' stroke-width='2' marker-end='url(#a)'/>
    <text x='317' y='28' font-size='10' fill='#555' font-family='Arial'>rows</text>
    <rect x='350' y='20' width='120' height='32' rx='6' fill='#27ae60'/>
    <text x='410' y='41' text-anchor='middle' fill='white' font-size='12' font-family='Arial'>pandas / results</text>
    <defs><marker id='a' markerWidth='8' markerHeight='8' refX='6' refY='3' orient='auto'>
      <path d='M0,0 L0,6 L8,3 z' fill='#333'/></marker></defs>
  </svg>
</div>
'''))


In [5]:
# Connect to MySQL — prints the connection object if successful
try:
    with connect(
        host="localhost",
        user=input("Username: "),
        password=getpass("Password: "),
        database="school_analytics",   # must already exist
    ) as connection:
        print("✅ Connected:", connection)

except Error as e:
    print(f"❌ {e}")


Username:  root
Password:  ········


✅ Connected: <mysql.connector.connection_cext.CMySQLConnection object at 0x129fef770>


## 1.3 · Create the School Database

Connect *without* specifying a database first — it does not exist yet.


In [6]:
# No database= argument — we are creating it from scratch
try:
    with connect(
        host="localhost",
        user=input("Username: "),
        password=getpass("Password: "),
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute("CREATE DATABASE IF NOT EXISTS school_analytics")
        print("✅ school_analytics database created")

except Error as e:
    print(f"❌ {e}")


Username:  root
Password:  ········


✅ school_analytics database created


In [7]:
# Confirm which databases now exist on the server
try:
    with connect(
        host="localhost",
        user=input("Username: "),
        password=getpass("Password: "),
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute("SHOW DATABASES")
            print("Databases on this server:")
            for (db,) in cursor:          # each row is a 1-tuple
                print("  •", db)

except Error as e:
    print(f"❌ {e}")


Username:  root
Password:  ········


Databases on this server:
  • bookshop
  • bookstore_db
  • example2
  • ines
  • information_schema
  • mydb
  • mysql
  • performance_schema
  • school_analytics
  • sys


## 1.4 · Schema Design — Four Related Tables

`grades` is the central **fact table**; the other three are **dimension tables**.
Foreign keys ensure every grade row points to a valid student, subject and teacher.


In [8]:
from IPython.display import HTML, display
display(HTML('''
<div style='margin:10px 0'>
<svg width='540' height='220' xmlns='http://www.w3.org/2000/svg'
     style='font-family:Arial,sans-serif;font-size:12px;background:#fafafa;
            border-radius:10px;border:1px solid #ddd'>

  <!-- teachers -->
  <rect x='20'  y='80'  width='130' height='90' rx='8' fill='#EBF2FF' stroke='#4C72B0' stroke-width='1.5'/>
  <rect x='20'  y='80'  width='130' height='22' rx='8' fill='#4C72B0'/>
  <text x='85'  y='96'  text-anchor='middle' fill='white' font-weight='bold'>teachers</text>
  <text x='30'  y='118' fill='#333'>🔑 teacher_id</text>
  <text x='30'  y='133' fill='#555'>   first_name</text>
  <text x='30'  y='148' fill='#555'>   department</text>
  <text x='30'  y='163' fill='#555'>   years_exp</text>

  <!-- students -->
  <rect x='200' y='10'  width='140' height='100' rx='8' fill='#EDFBF0' stroke='#55A868' stroke-width='1.5'/>
  <rect x='200' y='10'  width='140' height='22'  rx='8' fill='#55A868'/>
  <text x='270' y='26'  text-anchor='middle' fill='white' font-weight='bold'>students</text>
  <text x='210' y='48'  fill='#333'>🔑 student_id</text>
  <text x='210' y='63'  fill='#555'>   first_name</text>
  <text x='210' y='78'  fill='#555'>   year_group</text>
  <text x='210' y='93'  fill='#555'>   house</text>

  <!-- subjects -->
  <rect x='390' y='80'  width='130' height='90' rx='8' fill='#F3EEFF' stroke='#8172B2' stroke-width='1.5'/>
  <rect x='390' y='80'  width='130' height='22' rx='8' fill='#8172B2'/>
  <text x='455' y='96'  text-anchor='middle' fill='white' font-weight='bold'>subjects</text>
  <text x='400' y='118' fill='#333'>🔑 subject_id</text>
  <text x='400' y='133' fill='#555'>   subject_name</text>
  <text x='400' y='148' fill='#555'>   department</text>
  <text x='400' y='163' fill='#555'>   is_core</text>

  <!-- grades -->
  <rect x='180' y='145' width='180' height='60' rx='8' fill='#FFF0F0' stroke='#C44E52' stroke-width='2'/>
  <rect x='180' y='145' width='180' height='22' rx='8' fill='#C44E52'/>
  <text x='270' y='161' text-anchor='middle' fill='white' font-weight='bold'>grades (fact table)</text>
  <text x='190' y='183' fill='#333'>🔗 student_id · subject_id</text>
  <text x='190' y='198' fill='#333'>🔗 teacher_id · percentage</text>

  <!-- arrows -->
  <defs><marker id='arr' markerWidth='8' markerHeight='8' refX='6' refY='3' orient='auto'>
    <path d='M0,0 L0,6 L8,3 z' fill='#888'/></marker></defs>
  <line x1='150' y1='140' x2='215' y2='155' stroke='#888' stroke-width='1.8' marker-end='url(#arr)'/>
  <line x1='270' y1='110' x2='270' y2='145' stroke='#888' stroke-width='1.8' marker-end='url(#arr)'/>
  <line x1='390' y1='140' x2='330' y2='155' stroke='#888' stroke-width='1.8' marker-end='url(#arr)'/>
  <text x='158' y='152' fill='#888' font-size='10'>FK</text>
  <text x='274' y='132' fill='#888' font-size='10'>FK</text>
  <text x='348' y='152' fill='#888' font-size='10'>FK</text>
</svg></div>
'''))


## 1.5 · Create the Four Tables

`CREATE TABLE IF NOT EXISTS` is safe to re-run — it never overwrites existing data.


In [9]:
# ── students: one row per pupil ─────────────────────────────────
create_students = """
CREATE TABLE IF NOT EXISTS students (
    student_id    VARCHAR(10) PRIMARY KEY,
    first_name    VARCHAR(50) NOT NULL,
    last_name     VARCHAR(50) NOT NULL,
    year_group    TINYINT     NOT NULL,
    gender        CHAR(1),
    house         VARCHAR(30),
    tutor_group   VARCHAR(5),
    entry_year    YEAR,
    pupil_premium CHAR(1)     DEFAULT 'N',
    sen_support   VARCHAR(30) DEFAULT 'None',
    form_tutor_id VARCHAR(10)
)"""

# ── subjects: lookup table (10 subjects) ─────────────────────────
create_subjects = """
CREATE TABLE IF NOT EXISTS subjects (
    subject_id   VARCHAR(5)  PRIMARY KEY,
    subject_name VARCHAR(50) NOT NULL,
    department   VARCHAR(30),
    key_stage    TINYINT,
    is_core      BOOLEAN     DEFAULT FALSE
)"""

print("students & subjects DDL defined")


students & subjects DDL defined


In [11]:
# ── teachers: staff details ───────────────────────────────────────
create_teachers = """
CREATE TABLE IF NOT EXISTS teachers (
    teacher_id    VARCHAR(10) PRIMARY KEY,
    first_name    VARCHAR(50) NOT NULL,
    last_name     VARCHAR(50) NOT NULL,
    subject_id    VARCHAR(5),
    department    VARCHAR(30),
    years_exp     TINYINT,
    is_form_tutor BOOLEAN     DEFAULT FALSE,
    email         VARCHAR(80)
)"""

# ── grades: central fact table with foreign key constraints ───────
create_grades = """
CREATE TABLE IF NOT EXISTS grades (
    record_id       INT         PRIMARY KEY AUTO_INCREMENT,
    student_id      VARCHAR(10) NOT NULL,
    subject_id      VARCHAR(5)  NOT NULL,
    teacher_id      VARCHAR(10),
    academic_year   VARCHAR(10),
    term            VARCHAR(10),
    percentage      DECIMAL(5,2),
    grade           VARCHAR(3),
    effort          TINYINT,
    predicted_grade VARCHAR(3),
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (subject_id) REFERENCES subjects(subject_id),
    FOREIGN KEY (teacher_id) REFERENCES teachers(teacher_id)
)"""

print("teachers & grades DDL defined")


teachers & grades DDL defined


In [12]:
# Execute all four CREATE statements inside one connection
try:
    with connect(
        host="localhost",
        user=input("Username: "),
        password=getpass("Password: "),
        database="school_analytics",
    ) as connection:
        with connection.cursor() as cursor:
            for sql in [create_students, create_subjects,
                        create_teachers, create_grades]:
                cursor.execute(sql)    # run each CREATE statement
        connection.commit()
        print("✅ All 4 tables created")

except Error as e:
    print(f"❌ {e}")


Username:  root
Password:  ········


✅ All 4 tables created


## 1.6 · Load School Data with pandas

Always explore your CSV files *before* loading them into a database.
This catches type mismatches and missing values early.


In [13]:
from IPython.display import HTML, display
display(HTML('''
<div style='background:#f0fff4;border:1px solid #55A868;border-radius:8px;
            padding:12px 18px;margin:6px 0;font-family:Arial,sans-serif'>
  <b style='color:#2d6a4f'>📂 Workflow: CSV → pandas → MySQL</b>
  <svg xmlns='http://www.w3.org/2000/svg' width='500' height='60' style='display:block;margin-top:8px'>
    <rect x='0'   y='14' width='90'  height='30' rx='6' fill='#55A868'/>
    <text x='45'  y='34' text-anchor='middle' fill='white' font-size='11' font-family='Arial'>students.csv</text>
    <line x1='90' y1='29' x2='135' y2='29' stroke='#333' stroke-width='1.5' marker-end='url(#b)'/>
    <rect x='135' y='14' width='100' height='30' rx='6' fill='#3B6AC4'/>
    <text x='185' y='34' text-anchor='middle' fill='white' font-size='11' font-family='Arial'>pd.read_csv()</text>
    <line x1='235' y1='29' x2='280' y2='29' stroke='#333' stroke-width='1.5' marker-end='url(#b)'/>
    <rect x='280' y='14' width='100' height='30' rx='6' fill='#e67e22'/>
    <text x='330' y='34' text-anchor='middle' fill='white' font-size='11' font-family='Arial'>clean &amp; build rows</text>
    <line x1='380' y1='29' x2='420' y2='29' stroke='#333' stroke-width='1.5' marker-end='url(#b)'/>
    <rect x='420' y='14' width='75'  height='30' rx='6' fill='#C44E52'/>
    <text x='457' y='34' text-anchor='middle' fill='white' font-size='11' font-family='Arial'>MySQL</text>
    <defs><marker id='b' markerWidth='7' markerHeight='7' refX='5' refY='3' orient='auto'>
      <path d='M0,0 L0,6 L7,3 z' fill='#333'/></marker></defs>
  </svg>
</div>
'''))


In [14]:
# Load all three CSV files — adjust the path if needed
df_students = pd.read_csv("students.csv")
df_teachers = pd.read_csv("teachers.csv")
df_grades   = pd.read_csv("grades.csv")

print(f"students : {df_students.shape[0]} rows × {df_students.shape[1]} cols")
print(f"teachers : {df_teachers.shape[0]} rows × {df_teachers.shape[1]} cols")
print(f"grades   : {df_grades.shape[0]} rows × {df_grades.shape[1]} cols")


students : 30 rows × 11 cols
teachers : 14 rows × 8 cols
grades   : 1620 rows × 10 cols


In [15]:
# Preview the students data — check column names and types
df_students.head()


,student_id,first_name,last_name,year_group,gender,house,tutor_group,entry_year,pupil_premium,sen_support,form_tutor_id
0,S001,Amelia,Smith,7,M,Attenborough,7A,2023,N,NaN,T03
1,S002,Noah,Johnson,7,F,Churchill,7B,2023,N,NaN,T03
2,S003,Olivia,Williams,7,M,Darwin,7C,2023,N,NaN,T03
3,S004,Liam,Brown,7,F,Newton,7D,2023,N,Dyslexia,T03
4,S005,Isla,Taylor,7,M,Attenborough,7A,2023,Y,NaN,T03


In [16]:
# Check for missing values — handle these before inserting into MySQL
print("Missing values:")
print(df_students.isnull().sum())


Missing values:
student_id        0
first_name        0
last_name         0
year_group        0
gender            0
house             0
tutor_group       0
entry_year        0
pupil_premium     0
sen_support      24
form_tutor_id     0
dtype: int64


In [17]:
# Quick summary statistics for the grades numeric columns
df_grades[["percentage", "effort"]].describe().round(2)


,percentage,effort
count,1562.00,1562.00
mean,64.11,3.52
std,16.92,1.06
min,15.00,1.00
25%,52.00,3.00
50%,64.00,4.00
75%,77.00,4.00
max,100.00,5.00


## 1.7 · Prepare Data Rows for Insertion

Build all row tuples in Python *before* opening the database connection.
This keeps the database block short and focused.


In [18]:
# ── subjects: hand-coded lookup rows ────────────────────────────
subjects_rows = [
    ("SUB01","Mathematics",      "STEM",       4, True),
    ("SUB02","English Language", "Humanities", 4, True),
    ("SUB03","Biology",          "Science",    4, False),
    ("SUB04","Chemistry",        "Science",    4, False),
    ("SUB05","Physics",          "Science",    4, False),
    ("SUB06","History",          "Humanities", 4, False),
    ("SUB07","Geography",        "Humanities", 4, False),
    ("SUB08","French",           "Languages",  4, False),
    ("SUB09","Computing",        "STEM",       4, False),
    ("SUB10","Art & Design",     "Creative",   4, False),
]
print(f"{len(subjects_rows)} subject rows ready")


10 subject rows ready


In [19]:
# ── students: convert DataFrame rows to list of tuples ───────────
student_rows = [
    tuple(row) for row in df_students[[
        "student_id","first_name","last_name","year_group","gender",
        "house","tutor_group","entry_year","pupil_premium",
        "sen_support","form_tutor_id"
    ]].values
]
print(f"{len(student_rows)} student rows ready")


30 student rows ready


In [20]:
# ── subject name → ID map (needed when inserting teachers/grades) ─
subj_map = {
    "Mathematics":"SUB01", "English Language":"SUB02", "Biology":"SUB03",
    "Chemistry":"SUB04",   "Physics":"SUB05",   "History":"SUB06",
    "Geography":"SUB07",   "French":"SUB08",    "Computing":"SUB09",
    "Art & Design":"SUB10","English Literature":"SUB02","RE":"SUB06","PE":"SUB10",
}

# ── teachers: map subject name to ID and convert is_form_tutor ────
teacher_rows = [
    (
        r["teacher_id"], r["first_name"], r["last_name"],
        subj_map.get(r["subject"], "SUB01"),  # name → ID
        r["department"], r["years_experience"],
        1 if r["is_form_tutor"] == "Yes" else 0,
        r["email"]
    )
    for _, r in df_teachers.iterrows()
]
print(f"{len(teacher_rows)} teacher rows ready")


14 teacher rows ready


In [27]:
# ── grades: handle NaN safely before inserting ───────────────────
import math

def safe(val):
    """Return None for NaN; MySQL stores None as SQL NULL."""
    return None if isinstance(val, float) and math.isnan(val) else val

grade_rows = [
    (
        r["student_id"],
        subj_map.get(r["subject"], "SUB01"),  # subject name → ID
        r["teacher_id"],
        r["academic_year"], r["term"],
        safe(r["percentage"]), safe(r["grade"]),
        safe(r["effort"]),     safe(r["predicted_grade"])
    )
    for _, r in df_grades.iterrows()
]
print(f"{len(grade_rows):,} grade rows ready")


1,620 grade rows ready


## 1.8 · Insert Data — One Table at a Time

Each table gets its own cell and its own `commit()`.  
This releases locks quickly and makes it easy to spot which table caused an error.

> ⚠️ **Insert order matters:** subjects → students → teachers → grades  
> (grades has foreign keys pointing to the other three, so they must exist first)


In [22]:
from IPython.display import HTML, display
display(HTML('''
<div style='background:#fff8e1;border:1px solid #f0c040;border-radius:8px;padding:10px 16px'>
  <b>📌 Insert order (foreign key dependency)</b>
  <svg xmlns='http://www.w3.org/2000/svg' width='500' height='44'>
    <rect x='0'   y='7' width='100' height='28' rx='5' fill='#8172B2'/>
    <text x='50'  y='26' text-anchor='middle' fill='white' font-size='12' font-family='Arial'>subjects</text>
    <line x1='100' y1='21' x2='130' y2='21' stroke='#555' stroke-width='1.5' marker-end='url(#c)'/>
    <rect x='130' y='7' width='100' height='28' rx='5' fill='#55A868'/>
    <text x='180' y='26' text-anchor='middle' fill='white' font-size='12' font-family='Arial'>students</text>
    <line x1='230' y1='21' x2='260' y2='21' stroke='#555' stroke-width='1.5' marker-end='url(#c)'/>
    <rect x='260' y='7' width='100' height='28' rx='5' fill='#4C72B0'/>
    <text x='310' y='26' text-anchor='middle' fill='white' font-size='12' font-family='Arial'>teachers</text>
    <line x1='360' y1='21' x2='390' y2='21' stroke='#555' stroke-width='1.5' marker-end='url(#c)'/>
    <rect x='390' y='7' width='100' height='28' rx='5' fill='#C44E52'/>
    <text x='440' y='26' text-anchor='middle' fill='white' font-size='12' font-family='Arial'>grades</text>
    <defs><marker id='c' markerWidth='7' markerHeight='7' refX='5' refY='3' orient='auto'>
      <path d='M0,0 L0,6 L7,3 z' fill='#555'/></marker></defs>
  </svg>
</div>
'''))


In [25]:
# ── Step 1: Insert subjects (10 rows — safe to do in one go) ─────
try:
    with connect(host="localhost", user=input("Username: "),
                 password=getpass("Password: "),
                 database="school_analytics") as connection:
        with connection.cursor() as cursor:
            for row in subjects_rows:            # one row at a time
                cursor.execute(
                    "INSERT IGNORE INTO subjects VALUES (%s,%s,%s,%s,%s)", row)
                connection.commit()              # release lock after each row
        print(f"✅ {len(subjects_rows)} subjects inserted")

except Error as e:
    print(f"❌ {e}")


Username:  root
Password:  ········


✅ 10 subjects inserted


In [29]:
# replace nan with None before inserting
import math

student_rows = [
    tuple(None if isinstance(v, float) and math.isnan(v) else v for v in row)
    for row in student_rows
]

In [30]:
# ── Step 2: Insert students ──────────────────────────────────────
try:
    with connect(host="localhost", user=input("Username: "),
                 password=getpass("Password: "),
                 database="school_analytics") as connection:
        with connection.cursor() as cursor:
            for row in student_rows:
                cursor.execute(
                    "INSERT IGNORE INTO students VALUES "
                    "(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)", row)
                connection.commit()
        print(f"✅ {len(student_rows)} students inserted")

except Error as e:
    print(f"❌ {e}")


Username:  root
Password:  ········


✅ 30 students inserted


In [31]:
# ── Step 3: Insert teachers ──────────────────────────────────────
try:
    with connect(host="localhost", user=input("Username: "),
                 password=getpass("Password: "),
                 database="school_analytics") as connection:
        with connection.cursor() as cursor:
            for row in teacher_rows:
                cursor.execute(
                    "INSERT IGNORE INTO teachers VALUES (%s,%s,%s,%s,%s,%s,%s,%s)", row)
                connection.commit()
        print(f"✅ {len(teacher_rows)} teachers inserted")

except Error as e:
    print(f"❌ {e}")


Username:  root
Password:  ········


✅ 14 teachers inserted


In [32]:
# ── Step 4: Insert grades in batches of 200 ──────────────────────
# Large tables are inserted in small chunks to avoid lock timeouts
BATCH = 200

try:
    with connect(host="localhost", user=input("Username: "),
                 password=getpass("Password: "),
                 database="school_analytics") as connection:
        with connection.cursor() as cursor:
            for i in range(0, len(grade_rows), BATCH):
                chunk = grade_rows[i : i + BATCH]  # slice 200 rows
                cursor.executemany(
                    "INSERT IGNORE INTO grades "
                    "(student_id,subject_id,teacher_id,academic_year,term,"
                    "percentage,grade,effort,predicted_grade) "
                    "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)", chunk)
                connection.commit()               # commit each batch
                print(f"  batch {i//BATCH+1}: rows {i+1}–{i+len(chunk)}")

        print(f"✅ {len(grade_rows):,} grades inserted")

except Error as e:
    print(f"❌ {e}")


Username:  root
Password:  ········


  batch 1: rows 1–200
  batch 2: rows 201–400
  batch 3: rows 401–600
  batch 4: rows 601–800
  batch 5: rows 801–1000
  batch 6: rows 1001–1200
  batch 7: rows 1201–1400
  batch 8: rows 1401–1600
  batch 9: rows 1601–1620
✅ 1,620 grades inserted


## 1.9 · Querying Data with SELECT

`pd.read_sql()` runs a query and returns a pandas DataFrame in one line —
much more convenient than looping through cursor rows.


In [33]:
# Store credentials once for the session
DB = dict(
    host="localhost",
    user=input("Username: "),
    password=getpass("Password: "),
    database="school_analytics"
)
print("✅ Credentials ready")


Username:  root
Password:  ········


✅ Credentials ready


In [38]:
# Install if needed: pip install sqlalchemy

# SELECT all students
from sqlalchemy import create_engine

engine = create_engine(
    f"mysql+mysqlconnector://{DB['user']}:{DB['password']}@{DB['host']}/{DB['database']}"
)

with engine.connect() as conn:
    df = pd.read_sql("SELECT * FROM students LIMIT 5", conn)

df

,student_id,first_name,last_name,year_group,gender,house,tutor_group,entry_year,pupil_premium,sen_support,form_tutor_id
0,S001,Amelia,Smith,7,M,Attenborough,7A,2023,N,None,T03
1,S002,Noah,Johnson,7,F,Churchill,7B,2023,N,None,T03
2,S003,Olivia,Williams,7,M,Darwin,7C,2023,N,None,T03
3,S004,Liam,Brown,7,F,Newton,7D,2023,N,Dyslexia,T03
4,S005,Isla,Taylor,7,M,Attenborough,7A,2023,Y,None,T03


In [39]:
# SELECT specific columns — only name and year group
with connect(**DB) as conn:
    df = pd.read_sql(
        "SELECT first_name, last_name, year_group, house FROM students",
        conn)
df.head()


/var/folders/f0/4_8w17g93v712z_98z2gdvg40000gn/T/ipykernel_1639/3418433564.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,first_name,last_name,year_group,house
0,Amelia,Smith,7,Attenborough
1,Noah,Johnson,7,Churchill
2,Olivia,Williams,7,Darwin
3,Liam,Brown,7,Newton
4,Isla,Taylor,7,Attenborough


In [40]:
# WHERE clause — pupils in Year 11 only
with connect(**DB) as conn:
    df = pd.read_sql(
        "SELECT first_name, last_name, house FROM students "
        "WHERE year_group = 11 ORDER BY last_name",
        conn)
df


/var/folders/f0/4_8w17g93v712z_98z2gdvg40000gn/T/ipykernel_1639/2532794710.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,first_name,last_name,house
0,Muhammad,Adams,Churchill
1,Jasmine,Baker,Darwin
2,Ruby,Green,Attenborough
3,Archie,Hill,Newton
4,Poppy,Jackson,Attenborough
5,Reuben,Scott,Churchill


In [41]:
# JOIN — combine grades with student names and subject names
with connect(**DB) as conn:
    df = pd.read_sql("""
        SELECT
            CONCAT(st.first_name," ",st.last_name) AS pupil,
            sub.subject_name,
            g.academic_year,
            g.term,
            g.percentage,
            g.grade
        FROM   grades   g
        JOIN   students st  ON g.student_id  = st.student_id
        JOIN   subjects sub ON g.subject_id  = sub.subject_id
        WHERE  g.percentage IS NOT NULL
        ORDER  BY g.percentage DESC
        LIMIT  10
    """, conn)
df


/var/folders/f0/4_8w17g93v712z_98z2gdvg40000gn/T/ipykernel_1639/3545939167.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


,pupil,subject_name,academic_year,term,percentage,grade
0,Muhammad Adams,Computing,2024-25,Autumn,100.0,A*
1,Jasmine Baker,Computing,2023-24,Summer,100.0,A*
2,Muhammad Adams,Chemistry,2024-25,Autumn,100.0,A*
3,Muhammad Adams,Biology,2023-24,Spring,100.0,A*
4,Theo White,Physics,2024-25,Summer,100.0,A*
5,Chloe Thompson,Mathematics,2023-24,Autumn,100.0,A*
6,Chloe Thompson,Physics,2022-23,Summer,100.0,A*
7,Jasmine Baker,Chemistry,2022-23,Spring,100.0,A*
8,Muhammad Adams,Chemistry,2023-24,Autumn,100.0,A*
9,Muhammad Adams,English Language,2023-24,Autumn,100.0,A*


In [42]:
# GROUP BY — average grade per subject (like a Pivot Table)
with connect(**DB) as conn:
    df = pd.read_sql("""
        SELECT
            sub.subject_name,
            ROUND(AVG(g.percentage),1) AS avg_pct,
            COUNT(*) AS entries
        FROM   grades   g
        JOIN   subjects sub ON g.subject_id = sub.subject_id
        WHERE  g.percentage IS NOT NULL
        GROUP  BY sub.subject_name
        ORDER  BY avg_pct DESC
    """, conn)
df


/var/folders/f0/4_8w17g93v712z_98z2gdvg40000gn/T/ipykernel_1639/1998369617.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


,subject_name,avg_pct,entries
0,Chemistry,65.3,158
1,Computing,64.7,158
2,French,64.6,158
3,Art & Design,64.6,157
4,English Language,64.1,157
5,Mathematics,63.8,159
6,History,63.8,156
7,Biology,63.7,154
8,Physics,63.4,154
9,Geography,63.0,151


## 1.10 · Updating a Record

Use `UPDATE` when a value changes — for example when a pupil's SEN status is revised.


In [43]:
# Update a single student's SEN support field
try:
    with connect(**DB) as connection:
        with connection.cursor() as cursor:
            cursor.execute(
                "UPDATE students SET sen_support = %s WHERE student_id = %s",
                ("EAL", "S005")           # new value, target row
            )
        connection.commit()
        print(f"✅ {cursor.rowcount} row updated")

except Error as e:
    print(f"❌ {e}")


✅ 1 row updated


In [44]:
# Verify the update worked
with connect(**DB) as conn:
    df = pd.read_sql(
        "SELECT student_id, first_name, sen_support FROM students "
        "WHERE student_id = 'S005'", conn)
df


/var/folders/f0/4_8w17g93v712z_98z2gdvg40000gn/T/ipykernel_1639/1995060620.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,student_id,first_name,sen_support
0,S005,Isla,EAL


## 1.11 · Deleting a Record

Always use a `WHERE` clause with DELETE — without it you will wipe the entire table.


In [45]:
# Example: remove a test/dummy grade entry
try:
    with connect(**DB) as connection:
        with connection.cursor() as cursor:
            cursor.execute(
                "DELETE FROM grades WHERE grade = %s AND percentage < %s",
                ("U", 20)                 # only remove clearly erroneous entries
            )
        connection.commit()
        print(f"✅ {cursor.rowcount} row(s) deleted")

except Error as e:
    print(f"❌ {e}")


✅ 2 row(s) deleted


## 1.12 · Troubleshooting Lock Timeouts

If you see `Error 1205: Lock wait timeout exceeded`, run the cell below  
to see which connections are blocking, then kill them in MySQL Workbench  
(**Server → Client Connections → Kill Connection**).


In [46]:
# Show all active connections on the server
try:
    with connect(
        host="localhost",
        user=input("Username: "),
        password=getpass("Password: "),
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute("SHOW PROCESSLIST")
            print(f"{"ID":<6} {"User":<10} {"DB":<20} {"Command":<10} {"Time":>5} Info")
            print("-" * 70)
            for row in cursor.fetchall():
                print(f"{str(row[0]):<6} {str(row[1]):<10} {str(row[3]):<20}"
                      f" {str(row[4]):<10} {str(row[5]):>5} {str(row[7])[:30]}")

except Error as e:
    print(f"❌ {e}")


Username:  root
Password:  ········


ID     User       DB                   Command     Time Info
----------------------------------------------------------------------
5      event_scheduler None                 Daemon       776 None
23     root       school_analytics     Sleep        143 None
25     root       school_analytics     Sleep         84 None
33     root       None                 Query          0 SHOW PROCESSLIST


In [47]:
from IPython.display import HTML, display
display(HTML('''
<div style='display:flex;align-items:center;gap:16px;padding:14px 20px;
            background:#f0fff8;border-left:6px solid #27ae60;border-radius:8px;margin:16px 0'>
  <svg xmlns='http://www.w3.org/2000/svg' width='48' height='48' viewBox='0 0 48 48'>
    <rect width='48' height='48' rx='8' fill='#27ae60'/>
    <text x='24' y='34' font-size='26' text-anchor='middle' fill='white' font-family='Arial'>💾</text>
  </svg>
  <div>
    <div style='font-size:22px;font-weight:bold;color:#1a1a2e'>Part 2 — SQLite</div>
    <div style='font-size:13px;color:#555;margin-top:2px'>
      Built into Python · No server · Single file · Perfect for quick local analysis
    </div>
  </div>
</div>
'''))


## 2.1 · What is SQLite?

SQLite stores the entire database in a single `.db` file on disk.  
There is no server to install — `import sqlite3` is all you need.

| | MySQL | SQLite |
|--|-------|--------|
| Server | Required | ❌ None |
| Install | Yes | Built into Python |
| Best for | School MIS, multi-user | Local scripts, prototyping |
| File | Many files | One `.db` file |


In [48]:
import sqlite3
import pandas as pd

# Creates the file if it does not exist, opens it if it does
conn = sqlite3.connect("school_lite.db")
cursor = conn.cursor()

print("✅ Connected to school_lite.db")


✅ Connected to school_lite.db


## 2.2 · Create a Pastoral Events Table

We'll track reward and sanction events — a common school MIS feature.


In [49]:
# Create the table — note SQLite uses INTEGER PRIMARY KEY for auto-increment
cursor.execute("""
    CREATE TABLE IF NOT EXISTS pastoral_events (
        event_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        student_id TEXT    NOT NULL,
        event_type TEXT,          -- "Reward" or "Sanction"
        category   TEXT,          -- e.g. "Excellent Work"
        event_date TEXT,          -- stored as ISO string "YYYY-MM-DD"
        points     INTEGER DEFAULT 0
    )
""")
conn.commit()
print("✅ pastoral_events table ready")


✅ pastoral_events table ready


## 2.3 · Insert Pastoral Records

SQLite uses `?` as its placeholder — not `%s` like MySQL.


In [50]:
# Sample pastoral data — rewards and sanctions
events = [
    ("S001","Reward",   "Excellent Work",    "2024-09-10",  3),
    ("S002","Sanction", "Late to lesson",    "2024-09-11", -1),
    ("S003","Reward",   "Kindness Award",    "2024-09-12",  2),
    ("S004","Sanction", "Mobile phone use",  "2024-09-13", -2),
    ("S005","Reward",   "100% Attendance",   "2024-09-14",  5),
    ("S001","Reward",   "Subject Prize",     "2024-10-01",  4),
    ("S002","Reward",   "Effort Award",      "2024-10-05",  3),
    ("S004","Sanction", "Uniform",           "2024-10-08", -1),
    ("S006","Sanction", "Disruption",        "2024-10-09", -3),
    ("S007","Reward",   "Excellent Work",    "2024-10-15",  3),
]

# executemany with ? placeholders
cursor.executemany(
    "INSERT INTO pastoral_events "
    "(student_id,event_type,category,event_date,points) VALUES (?,?,?,?,?)",
    events
)
conn.commit()
print(f"✅ {len(events)} pastoral events inserted")


✅ 10 pastoral events inserted


## 2.4 · Query SQLite with pandas


In [51]:
# pd.read_sql works with SQLite connections exactly like MySQL
df_events = pd.read_sql(
    "SELECT * FROM pastoral_events ORDER BY event_date",
    conn
)
df_events


,event_id,student_id,event_type,category,event_date,points
0,1,S001,Reward,Excellent Work,2024-09-10,3
1,2,S002,Sanction,Late to lesson,2024-09-11,-1
2,3,S003,Reward,Kindness Award,2024-09-12,2
3,4,S004,Sanction,Mobile phone use,2024-09-13,-2
4,5,S005,Reward,100% Attendance,2024-09-14,5
5,6,S001,Reward,Subject Prize,2024-10-01,4
6,7,S002,Reward,Effort Award,2024-10-05,3
7,8,S004,Sanction,Uniform,2024-10-08,-1
8,9,S006,Sanction,Disruption,2024-10-09,-3
9,10,S007,Reward,Excellent Work,2024-10-15,3


In [52]:
# Net points per pupil — positive = more rewards than sanctions
df_net = pd.read_sql("""
    SELECT
        student_id,
        SUM(points)                         AS net_points,
        SUM(CASE WHEN points > 0 THEN 1 END) AS rewards,
        SUM(CASE WHEN points < 0 THEN 1 END) AS sanctions
    FROM pastoral_events
    GROUP BY student_id
    ORDER BY net_points DESC
""", conn)
df_net


,student_id,net_points,rewards,sanctions
0,S001,7,2.0,NaN
1,S005,5,1.0,NaN
2,S007,3,1.0,NaN
3,S003,2,1.0,NaN
4,S002,2,1.0,1.0
5,S006,-3,NaN,1.0
6,S004,-3,NaN,2.0


## 2.5 · Load a pandas DataFrame directly into SQLite

`.to_sql()` creates the table and inserts all rows in one call —  
the quickest way to persist any DataFrame.


In [53]:
# Write the students DataFrame straight into the SQLite database
df_students.to_sql(
    "students",       # table name to create
    conn,
    if_exists="replace",   # "replace" drops & recreates; "append" adds rows
    index=False            # do not write the pandas integer index
)

# Verify it landed
count = pd.read_sql("SELECT COUNT(*) AS n FROM students", conn).iloc[0,0]
print(f"✅ {count} student rows written to SQLite")


✅ 30 student rows written to SQLite


In [54]:
# Read back a filtered result — now queryable like any SQL table
df_yr10 = pd.read_sql(
    "SELECT first_name, last_name, house FROM students WHERE year_group = 10",
    conn
)
print("Year 10 pupils:")
df_yr10


Year 10 pupils:


,first_name,last_name,house
0,Ava,Martin,Darwin
1,Theo,White,Newton
2,Chloe,Thompson,Attenborough
3,Oscar,Harris,Churchill
4,Lily,Lee,Darwin
5,Finn,Turner,Newton


In [55]:
# Always close the connection when finished
conn.close()
print("SQLite connection closed")


SQLite connection closed


In [56]:
from IPython.display import HTML, display
display(HTML('''
<div style='display:flex;align-items:center;gap:16px;padding:14px 20px;
            background:#f5f0ff;border-left:6px solid #6c3fa0;border-radius:8px;margin:16px 0'>
  <svg xmlns='http://www.w3.org/2000/svg' width='48' height='48' viewBox='0 0 48 48'>
    <rect width='48' height='48' rx='8' fill='#336791'/>
    <text x='24' y='34' font-size='26' text-anchor='middle' fill='white' font-family='Arial'>🐘</text>
  </svg>
  <div>
    <div style='font-size:22px;font-weight:bold;color:#1a1a2e'>Part 3 — PostgreSQL</div>
    <div style='font-size:13px;color:#555;margin-top:2px'>
      Enterprise-grade · Cloud databases (Heroku, Supabase, AWS RDS) · Same SQL, different connector
    </div>
  </div>
</div>
'''))


## 3.1 · Installation & Setup

```
pip install psycopg2-binary
```

Download PostgreSQL: https://www.postgresql.org/download/


In [ ]:
# psycopg2 is the standard PostgreSQL adapter for Python
# !pip install psycopg2-binary

import psycopg2
from getpass import getpass
import pandas as pd

print("psycopg2 ready")


## 3.2 · Connect to PostgreSQL

The connection call is similar to MySQL, but uses `psycopg2.connect()` and port 5432.


In [ ]:
# Open a connection — replace credentials with your own
try:
    conn = psycopg2.connect(
        host="localhost",
        port=5432,                        # default PostgreSQL port
        database="school_analytics",
        user=input("PG username: "),
        password=getpass("PG password: ")
    )
    print("✅ Connected to PostgreSQL")
    print(conn.get_dsn_parameters())      # connection properties

except Exception as e:
    print(f"❌ {e}")


## 3.3 · Create a Table

PostgreSQL uses `SERIAL` instead of MySQL's `AUTO_INCREMENT`.
Always call `rollback()` before retrying after an error.


In [ ]:
# Create an attendance tracking table
try:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS attendance (
                att_id     SERIAL PRIMARY KEY,
                student_id VARCHAR(10),
                att_date   DATE,
                present    BOOLEAN DEFAULT TRUE,
                session    VARCHAR(2) DEFAULT 'AM'
            )
        """)
    conn.commit()
    print("✅ attendance table created")

except Exception as e:
    conn.rollback()   # PostgreSQL requires explicit rollback on error
    print(f"❌ {e}")


## 3.4 · Insert Rows

PostgreSQL uses `%s` placeholders — same as MySQL.


In [ ]:
# Insert sample attendance records
attendance_data = [
    ("S001","2024-09-02",True,"AM"),
    ("S001","2024-09-02",True,"PM"),
    ("S002","2024-09-02",False,"AM"),  # absent morning
    ("S003","2024-09-02",True,"AM"),
    ("S003","2024-09-02",True,"PM"),
]

try:
    with conn.cursor() as cur:
        cur.executemany(
            "INSERT INTO attendance (student_id,att_date,present,session) "
            "VALUES (%s,%s,%s,%s)",
            attendance_data
        )
    conn.commit()
    print(f"✅ {len(attendance_data)} attendance rows inserted")

except Exception as e:
    conn.rollback()
    print(f"❌ {e}")


## 3.5 · Query with pandas


In [ ]:
# pd.read_sql works with psycopg2 connections exactly like MySQL
try:
    df_att = pd.read_sql(
        "SELECT * FROM attendance ORDER BY att_date, session",
        conn
    )
    df_att

except Exception as e:
    print(f"❌ {e}")


In [ ]:
# Absence rate per student (PostgreSQL FILTER syntax)
try:
    df_abs = pd.read_sql("""
        SELECT
            student_id,
            COUNT(*) AS total_sessions,
            SUM(CASE WHEN NOT present THEN 1 ELSE 0 END) AS absences,
            ROUND(100.0 * SUM(CASE WHEN NOT present THEN 1 ELSE 0 END)
                  / COUNT(*), 1) AS absence_pct
        FROM attendance
        GROUP BY student_id
        ORDER BY absence_pct DESC
    """, conn)
    print(df_abs)

except Exception as e:
    print(f"❌ {e}")

finally:
    conn.close()   # psycopg2 connections must be closed manually
    print("PostgreSQL connection closed")


## Summary — Three Databases Side by Side

| Feature | MySQL | SQLite | PostgreSQL |
|---------|-------|--------|------------|
| **Server** | Required | None (file) | Required |
| **Library** | `mysql-connector-python` | `sqlite3` (built-in) | `psycopg2` |
| **Connect** | `connect(host=…)` | `sqlite3.connect(file)` | `psycopg2.connect(…)` |
| **Placeholder** | `%s` | `?` | `%s` |
| **Auto-increment** | `AUTO_INCREMENT` | `AUTOINCREMENT` | `SERIAL` |
| **Rollback needed?** | No (with block) | No | Yes — explicit |
| **pandas** | `pd.read_sql(sql, conn)` | `pd.read_sql(sql, conn)` | `pd.read_sql(sql, conn)` |
| **Best for** | School MIS, web apps | Local tools, prototyping | Cloud, analytics, scale |

> 💡 **Key insight:** The SQL you write (`SELECT`, `JOIN`, `WHERE`, `GROUP BY`) is identical  
> across all three databases. Only the connection line and a few syntax details differ.
